In [ ]:
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Load data
X = pd.read_csv('/kaggle/input/titanic/train.csv', index_col='PassengerId')
X_test_full = pd.read_csv('/kaggle/input/titanic/test.csv', index_col='PassengerId')

y = X['Survived']
X.drop('Survived', axis=1, inplace=True)

# Feature Engineering
for df in [X, X_test_full]:
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
    df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)
    df['Fare'] = np.log1p(df['Fare'])
    df['TicketFreq'] = df.groupby('Ticket')['Ticket'].transform('count')


# Split
X_train_full, X_valid_full, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=0
)

# Column Management
low_cardinality_cols = [
    c for c in X_train_full.columns
    if X_train_full[c].dtype == 'object' and X_train_full[c].nunique() < 10
]
numeric_cols = X_train_full.select_dtypes(include=['int64', 'float64']).columns

my_cols = low_cardinality_cols + list(numeric_cols)

X_train = X_train_full[my_cols]
X_valid = X_valid_full[my_cols]
X_test = X_test_full[my_cols]

X_train = pd.get_dummies(X_train)
X_valid = pd.get_dummies(X_valid)
X_test = pd.get_dummies(X_test)

X_train, X_valid = X_train.align(X_valid, join='left', axis=1)
X_train, X_test = X_train.align(X_test, join='left', axis=1)

X_train.fillna(0, inplace=True)
X_valid.fillna(0, inplace=True)
X_test.fillna(0, inplace=True)

# Model Fitting
model = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=3,
    min_child_weight=1,
    subsample=0.8,
    early_stopping_rounds=20,
    colsample_bytree=0.8,
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=0
)

model.fit(
    X_train, y_train,
    eval_set=[(X_valid, y_valid)],
    verbose=False
)

# Evaluation
preds = model.predict(X_valid)
acc = accuracy_score(y_valid, preds)
print("Validation accuracy:", acc)

# Submission
preds_test = model.predict(X_test)
output = pd.DataFrame({'PassengerId': X_test.index, 'Survived': preds_test})
output.to_csv('submission.csv', index=False)
